# Exploring a GEOS cubed-sphere grid with pyramids-mesh

**GEOS** is NASA's Goddard Earth Observing System, an Earth-system model used for atmospheric reanalysis and forecasting. Instead of a single longitude/latitude grid (which crowds cells together at the poles), GEOS discretises the globe as a **cubed sphere**: a cube is inflated onto the sphere, and each of its **six panels** is tiled by an `N x N` array of cells. The resolution label `cN` is that `N` -- so this **C12** grid has `6 x 12 x 12 = 864` cells.

The native NetCDF-4 file stores only panel corners as `corner_lons`/`corner_lats` of shape `(6, N+1, N+1)`. The `geos` driver reconstructs the quadrilateral connectivity by index arithmetic across the six panels and hands back a pyramids `MeshDataset` -- after which nothing in the API remembers the file was a cubed sphere. We read the real UXarray C12 sample and tour what pyramids can tell us about it.

## Setup

`MeshDataset` is the object every pyramids-mesh driver returns -- a wrapper around a 2D or 1D mesh. Importing from `pyramids` also places its vendored GDAL/PROJ data on the path for us, so there is no import-order ritual in a notebook: we just import the names we use. `add_tiles` lets us drape a basemap *when* a grid carries a CRS (this one does not, as we will see).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

from pyramids.netcdf.ugrid import MeshDataset

## Locate the sample file

This notebook lives in `docs/examples/real-data/`, so the repository's `examples/data/real-data/` folder is **three** levels up. Paths inside a notebook are always written relative to the notebook itself, and pyramids' `read_file` takes a `pathlib.Path` directly -- we pass the `Path`, never `str(path)`.

In [ ]:
DATA = Path("../../../examples/data/real-data")
mesh_file = DATA / "geos" / "test-c12.native.nc4"
mesh_file, mesh_file.exists()

## Read the mesh

`MeshDataset.read_file(path, driver="geos")` dispatches the file to the `geos` driver, which reconstructs the connectivity and returns a `MeshDataset` -- the same type a native UGRID NetCDF file would produce. We name the driver explicitly because several formats share the `.nc4` suffix; auto-detection works, but being explicit is unambiguous.

In [ ]:
ds = MeshDataset.read_file(mesh_file, driver="geos")
type(ds).__name__

Reading a mesh is when cleopatra switches matplotlib to a headless backend. Right after the first read we let cleopatra pick the right backend for this environment -- the inline backend in a notebook, `Agg` in a plain script -- so every plot below embeds in the cell.

In [ ]:
from cleopatra.config import Config

Config.set_matplotlib_backend()

## Explore the dataset

A `MeshDataset` exposes summary properties for the whole mesh: `n_node` / `n_face` (size), `crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent as `xmin, ymin, xmax, ymax`), and `data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": ds.bounds,
    "data_variable_names": ds.data_variable_names,
})

**864 faces on 1014 nodes.** `crs` and `epsg` are `None` and `data_variable_names` is empty -- a native grid *description* carrying no CRS tag and no physical fields. The `bounds` already hint at the coordinate system: latitude runs the full `-90 -> 90` and longitude nearly the full circle.

## The coordinates are geographic lon/lat

GEOS corners are stored in **degrees** of longitude and latitude -- not a projected metric system and not radians. We check the node coordinate ranges directly rather than trust the format name.

In [ ]:
nx, ny = ds.mesh.node_x, ds.mesh.node_y
pd.Series({
    "node_x (lon) min": float(nx.min()),
    "node_x (lon) max": float(nx.max()),
    "node_y (lat) min": float(ny.min()),
    "node_y (lat) max": float(ny.max()),
})

Longitude spans roughly `0 -> 358` and latitude the full `-90 -> 90`: unmistakably geographic degrees on the whole sphere. Because `epsg` is `None`, pyramids will not *assume* this is EPSG:4326 -- a CRS-aware operation needs you to assign that CRS first (see the closing note). This is also why the basemap guard `if ds.epsg:` never fires below.

### Why 864 = 6 x 12 x 12

The connectivity table `face_node_connectivity` has shape `(n_face, k)`, where `k` is the number of nodes per face. Its two dimensions encode the whole cubed-sphere story at once.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
pd.Series({
    "connectivity_shape": conn.shape,
    "nodes_per_face (k)": conn.shape[1],
    "n_face": conn.shape[0],
    "6 * 12 * 12": 6 * 12 * 12,
})

The shape is **`(864, 4)`**: `k = 4` says every cell is a **quadrilateral**, and `864 = 6 x 12 x 12` is exactly the six cube panels at C12. The quad shape matters later -- some operations (circumcenters) are triangle-only.

## The wireframe

The simplest view needs no data: just draw the cell edges. `plot_outline()` renders the mesh skeleton through cleopatra and is the fastest check that a file read in correctly. We keep the `add_tiles` guard so a CRS-bearing grid would get a basemap -- here it is skipped because `epsg` is `None`.

In [ ]:
g = ds.plot_outline()
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

The six cube panels appear unrolled onto the lon/lat plane. The seams between panels are visible -- and are exactly why the connected-components count comes out as six at the end.

## A data map: cell latitude

A native GEOS grid carries no field values, so we **synthesise** one to demonstrate the data-map path. The centroid latitude of each cell is a natural choice -- it needs no external data and varies smoothly over the globe. `face_centroids` gives one `(x, y)` per face; we attach the `y` (latitude) as a face-located variable with `with_variable(...)`, which returns a **new** dataset and leaves `ds` untouched.

In [ ]:
cx, cy = ds.mesh.face_centroids
d2 = ds.with_variable("cell_latitude", cy, location="face")
d2.data_variable_names

Colouring by that field produces the generic pyramids data map -- the same call any real per-cell field (temperature, pressure, flow depth) would use. A diverging colormap reads naturally for latitude: cool at the south pole, warm at the north.

In [ ]:
g = d2.plot("cell_latitude", cmap="coolwarm", colorbar=True, title="Cell latitude (degrees)")
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

Each quad is shaded by its centroid latitude, giving a smooth south-to-north gradient across all six unrolled panels -- confirming the cells really do cover the whole globe and that the per-cell colouring works. `summary()` reports the statistics of every attached field.

In [ ]:
d2.summary()

The single `cell_latitude` row (count = 864) spans roughly `-84 -> 84` with a mean of ~0 -- symmetric about the equator, as a global grid should be (cell centres stop just short of the poles).

## GIS operations, for free

Because the mesh is a first-class pyramids object, the pyramids GIS layer applies directly -- none of these methods live in `pyramids-mesh`, which only parses the file. We tour three.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` -- the standard vector structure for export to shapefile/GeoPackage or for spatial joins.

In [ ]:
gdf = ds.to_geodataframe()
gdf.head()

One quadrilateral cell per row. The CRS column is unset (matching `epsg = None`); you would assign a geographic CRS before exporting for use in other GIS tools.

### Bounding polygon

`bounding_polygon()` dissolves the cells into their outer boundary. For the unrolled global C12 grid it returns a single shapely `Polygon` of the full extent.

In [ ]:
poly = ds.bounding_polygon()
pd.Series({"type": type(poly).__name__, "bounds": ds.bounds})

A plain shapely `Polygon` spanning the global degree extent -- ready for clipping or export.

### Connected components confirm one surface... or six

`connected_components()` labels each face by which connected piece of the face-adjacency graph it belongs to. The number of distinct labels tells you whether the mesh is one contiguous surface or several disjoint pieces.

In [ ]:
labels = ds.connected_components()
pd.Series({
    "n_components": int(np.unique(labels).size),
    "counts": np.bincount(labels).tolist(),
})

**Six** components of **144** cells each (`12 x 12`) -- one per cube panel. The driver stitches connectivity *within* each panel but not across the cube seams, so `connected_components` recovers the cubed sphere's six faces exactly. A geometry-revealing readout that falls straight out of the generic graph layer.

## What we cannot do here, and why

- **`to_crs(epsg)` is unavailable.** This grid records no CRS (`epsg` is `None`), so pyramids refuses to reproject -- there is no source to project *from*. You would first assign the known CRS (these global grids are geographic lon/lat on a sphere) and only then reproject.
- **No physical fields ship with the file.** The grid is pure geometry, so any data map (like the cell-latitude one above) must be synthesised or joined in from a separate dataset.

These notebooks also stay within pyramids-core's own GDAL-based UGRID I/O and do not reach for `xarray`, `h5py`, or other array libraries directly.

## Takeaway

- The `geos` driver reconstructs a GEOS cubed-sphere grid from its `(6, N+1, N+1)` corner arrays into a pyramids `MeshDataset`.
- C12 cells are **quadrilaterals** (`k = 4`, `864 = 6 x 12 x 12`) with coordinates in **geographic degrees** (lon `0 -> 358`, lat `-90 -> 90`), and the file carries **no CRS and no fields**.
- `connected_components` recovers the **six cube panels** (144 cells each) -- a direct readout of the cubed-sphere topology.
- Once read, the full pyramids GIS layer applies -- `to_geodataframe`, `bounding_polygon`, connected components, a synthesised data map -- while `to_crs` is skipped because no CRS was recorded.